直接用 results(dds)（不做shrinkage）的好处

是真正的最大似然估计(MLE)，无偏：results() 报出来的 log2FoldChange，本质上是GLM拟合出来的系数，没有被任何先验"拉拢"过。如果你的目的是做效应量的无偏估计（比如功效分析、跨研究meta分析、和别的工具结果比对），MLE比shrinkage后的估计更合适——shrinkage本质上是"用偏差换方差"（bias-variance tradeoff），牺牲了无偏性来换取更稳定/更可重复的数值。

更简单、更快、少一层"模型选择"的主观性：lfcShrink()要在 normal/apeglm/ashr 三种先验方法里选一个，不同方法收缩力度不同，等于多引入了一个需要解释的方法学选择；
直接用MLE省掉这一步。

不影响显著性判定：这点很关键——padj/p值是基于Wald检验，用的是MLE和它自己的标准误算出来的，从来不受shrinkage影响。
所以"谁是显著差异基因"这个判断，不管你做不做shrinkage，结果是完全一样的；唯一变化的是显著基因报告出来的log2FoldChange具体数值大小。



为什么、什么时候shrinkage成为主流
DESeq2 在2014年发表时（Love, Huber, Anders, Genome Biology）就内置了一种收缩机制：DESeq()函数有个参数叫 betaPrior，当时默认是TRUE——会给所有基因的log fold change统一套一个宽度固定的零均值正态先验，所以你直接调 results()，拿到的其实已经是收缩过的值，不是纯MLE。

<span style="color:magenta">所以你"几年前"的分析，如果用的是较老版本DESeq2，很可能在不知不觉中已经用了某种shrinkage，

只是没有显式写出lfcShrink()这一行。</span>



转折点大约在2016-2017年（Bioconductor 3.4 / DESeq2 1.16前后）：开发者把betaPrior的默认值改成了FALSE。原因是那种"所有基因统一收缩力度"的简单正态先验并不合理——表达量高、重复数多、本身估计就很稳的基因不该被收缩，而真正噪音大的基因（比如低表达、count接近0）才需要收缩；统一力度等于该收缩的没收够、不该收缩的反而被拉低了。

<span style="color:cyan">真正让lfcShrink()成为推荐标准做法的是2018年发表的apeglm方法（Zhu, Ibrahim & Love, Bioinformatics 2019，标题就叫"removing the noise and preserving large differences"）</span>


<span style="color:lightgreen"> ——它用近似后验的方式做自适应收缩：每个基因的收缩力度由它自己的标准误大小决定，噪音大的基因收缩得多，估计本身就很可靠的基因几乎不收缩。ashr（Stephens 2016, Biostatistics，就是你脚本里在用的这个）是另一种通用的经验贝叶斯自适应收缩方法，思路类似。从那以后，DESeq2官方vignette就明确推荐工作流程改成：先用results()做显著性判断（不受影响），再用lfcShrink()单独算一份"用于报告/排序/画图"的收缩后效应量——这也正是这个脚本（以及你当年那个没加shrinkage的版本，可能是基于更早期默认值的版本）所体现的两种不同时期的标准做法。</span>

结合你这次的opossum结果看：这恰好解释了之前我们诊断出的现象——ashr收缩之所以把几乎所有基因的log2FC压到接近0，是因为它自适应地判断"这批基因里真实有效应的比例很低"，所以收缩力度很大。这其实是自适应收缩方法诚实地反映数据本身信号弱，而不是bug；如果当年用的是纯MLE（或老版本DESeq2的统一弱先验），那12个|log2FC|>=1的基因可能只是低/中等表达量基因里被放大的噪音estimate，看起来"差异很大"但其实统计上并不可靠——这正是apeglm/ashr这类方法被设计出来要解决的痛点。